# **01: Data Exploration**

## **Objective**
Analyze the raw energy demand dataset to identify missing values, seasonality patterns, outliers, and data distributions to decide on preprocessing measures.

## **Dataset Info**
* **Source:** UCI Machine Learning Repository
* **Title:** Appliances Energy Prediction
* **Time Range:** 11th January 2016 - 27th May 2016
* **Primary Target Variable:** `Appliances` (Watt-hour consumed)

In [ ]:
# Import relevant packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from statsmodels.tsa.seasonal import STL
import os

## **Initial Inspection**
This section will inspect the dataset structure, statistics, characteristics and quality.

### Import dataset into pandas DataFrame

In [ ]:
# Get parent address
PARENT_ADDRESS = Path(os.getcwd()).parent

# Import csv file
energy = pd.read_csv(f'{PARENT_ADDRESS}/data/raw/energydata_complete.csv')

### Inspect samples

In [ ]:
energy.head(3)

### Inspect dataset characteristics

In [ ]:
# Print dataset info
energy.info()

### Check for Duplicated Index (date)

In [ ]:
# Check for duplicated date, as this will serve as index
print(f'Number of Duplicated Date: {energy['date'].duplicated().sum()}')

> ### 💡 **Key Takeaways**
> 1. **No missing values found**
>
> 2. **Date column have unconventional data type**
> 
>       **Decision**: Convert it to 'datetime64'and set as index.
>
> 3. **No duplicated index**

### Decision execution

In [ ]:
# Convert 'date' into datetime64 format
energy['date'] = energy['date'].astype('datetime64[s]')

# Set 'date' as index
energy = energy.set_index('date')

### Inspect changes

In [79]:
energy.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 19735 entries, 2016-01-11 17:00:00 to 2016-05-27 18:00:00
Data columns (total 22 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Appliances  19735 non-null  int64  
 1   lights      19735 non-null  int64  
 2   T1          19735 non-null  float64
 3   RH_1        19735 non-null  float64
 4   T2          19735 non-null  float64
 5   RH_2        19735 non-null  float64
 6   T3          19735 non-null  float64
 7   RH_3        19735 non-null  float64
 8   T4          19735 non-null  float64
 9   RH_4        19735 non-null  float64
 10  T5          19735 non-null  float64
 11  RH_5        19735 non-null  float64
 12  T6          19735 non-null  float64
 13  RH_6        19735 non-null  float64
 14  T7          19735 non-null  float64
 15  RH_7        19735 non-null  float64
 16  T8          19735 non-null  float64
 17  RH_8        19735 non-null  float64
 18  T9          19735 non-null  float64
 19  R

In [80]:
energy.describe()

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,T6,RH_6,T7,RH_7,T8,RH_8,T9,RH_9,rv1,rv2
count,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,...,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000
mean,97.694958,3.801875,21.686571,40.259739,20.341219,40.420420,22.267611,39.242500,20.855335,39.026904,...,7.910939,54.609083,20.267106,35.388200,22.029107,42.936165,19.485828,41.552401,24.988033,24.988033
std,102.524891,7.935988,1.606066,3.979299,2.192974,4.069813,2.006111,3.254576,2.042884,4.341321,...,6.090347,31.149806,2.109993,5.114208,1.956162,5.224361,2.014712,4.151497,14.496634,14.496634
min,10.000000,0.000000,16.790000,27.023333,16.100000,20.463333,17.200000,28.766667,15.100000,27.660000,...,-6.065000,1.000000,15.390000,23.200000,16.306667,29.600000,14.890000,29.166667,0.005322,0.005322
25%,50.000000,0.000000,20.760000,37.333333,18.790000,37.900000,20.790000,36.900000,19.530000,35.530000,...,3.626667,30.025000,18.700000,31.500000,20.790000,39.066667,18.000000,38.500000,12.497889,12.497889
50%,60.000000,0.000000,21.600000,39.656667,20.000000,40.500000,22.100000,38.530000,20.666667,38.400000,...,7.300000,55.290000,20.033333,34.863333,22.100000,42.375000,19.390000,40.900000,24.897653,24.897653
75%,100.000000,0.000000,22.600000,43.066667,21.500000,43.260000,23.290000,41.760000,22.100000,42.156667,...,11.256000,83.226667,21.600000,39.000000,23.390000,46.536000,20.600000,44.338095,37.583769,37.583769
max,1080.000000,70.000000,26.260000,63.360000,29.856667,56.026667,29.236000,50.163333,26.200000,51.090000,...,28.290000,99.900000,26.000000,51.400000,27.230000,58.780000,24.500000,53.326667,49.996530,49.996530


## **Drop Irrelevant/Unused Features**
We'll drop the features related to the readings taken from Chievres Airport, as data recorded from miles away introduces undesired spatial lag. Furthemore, we already have an outdoor temperature and humidity reading from the wall sensor (T6, RH_6).

In [ ]:
# Define unused columns
unused = [
    'T_out',
    'Press_mm_hg',
    'RH_out',
    'Windspeed',
    'Visibility',
    'Tdewpoint'
]

# Drop columns
energy = energy.drop(unused, axis = 1)

In [ ]:
# Check modified dataset
energy.dtypes

## **Outlier Detection Strategy**

> ### 1. Separate Features
For this part, we'll separate the dataset into 3 parts:
1. **Target feature** (Appliances)
2. **Temperature Feature** (T1 - T9)
3. **Humidity Feature** (RH_1 - RH_9)

**Justification**: Different scales, so features with smaller range and overall values tend to be flattened when plotted. For example, plotting Appliances (range ~1070 Wh) with temperature (range ~15 °C)  and humidity (range ~100 % )features in a time-series graph will flatten temperature the most. This will make visual inspection to detect spikes in temperature features much less prominent.

> ### 2. Aggregate by Rolling Mean
**Justification**: The dataset is sampled once every 10 minutes, which results in 144 samples each day. Plotting these raw values will result in a very noisy time-series plot.

> ### 3. 12-Hour as Window Size
**Justification**: The data was sampled in the course of 4 months, which is not long enough to reflect major seasonality change (winter, summer etc.). Hence, a 12-hour (or 72 samples) rolling window size is sufficient to reflect daily seasonality change (day and night) only.

### Visual Inspection

## **Aggregate Features**
Currently, the dataset contains 28 attributes, which can be complex to analyze. Furthermore, heat and moisture diffuse across adjacent room, which means that placing 28 sensors in a single house might be excessive and expensive. Hence, feature aggregation seems appropriate here.

We'll first look at feature collinearity to decide on the proper grouping. For visibility purposes, we'll divide the heatmap by temperature and humidity features.

### Determine Grouping Strategy

In [ ]:
# Define location sequence
loc = [
    "kitchen",
    "living room",
    "laundry room",
    "office room",
    "bathroom",
    "outside",
    "ironing room",
    "teenager room",
    "parents room",
]

# Initialize column names T1 - T9 and RH_1 - RH_9
temp_feature = [f"T{i+1}" for i in range(9)]
humid_feature = [f"RH_{i+1}" for i in range(9)]

# Calculate correlations
temp_corr = energy[temp_feature].corr(method="pearson")
humid_corr = energy[humid_feature].corr(method="pearson")

# Temperature Features
# ---
cg_temp = sns.clustermap(
    temp_corr,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    annot=False,
    linewidths=0.5,
    figsize=(4, 4),
    cbar_pos=(1.2, 0.2, 0.03, 0.4),
)

# Reorder locations based on temperature dendrogram
temp_labels = [loc[i] for i in cg_temp.dendrogram_col.reordered_ind]

# Set X and Y labels
cg_temp.ax_heatmap.set_xticklabels(temp_labels, rotation=90)
cg_temp.ax_heatmap.set_yticklabels(temp_labels, rotation=0)

# Set title
cg_temp.fig.suptitle(
    "Temperature Feature Correlation by Location", y=1.02, fontsize=12
)


# Humidity Features
# ---
cg_humid = sns.clustermap(
    humid_corr,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    annot=False,
    linewidths=0.5,
    figsize=(4, 4),
    cbar_pos=(1.2, 0.2, 0.03, 0.4),
)

# Reorder locations based on humidity dendrogram
humid_labels = [loc[i] for i in cg_humid.dendrogram_col.reordered_ind]

# Set X and Y labels
cg_humid.ax_heatmap.set_xticklabels(humid_labels, rotation=90)
cg_humid.ax_heatmap.set_yticklabels(humid_labels, rotation=0)

# Set title
cg_humid.fig.suptitle(
    "Humidity Feature Correlation by Location", y=1.02, fontsize=12
)

# Render both figures
plt.tight_layout()
plt.show()

Based on the heatmaps, this grouping seems appropriate:

### ```1. Main Indoor Areas``` 
> **Location**: kitchen, laundry room, ironing room, office room, teenager room, parents room

> **Justification**: High correlation across both temperature and humidity (signified by deep red color). Averaging eliminates severe multicollinearity with negligible loss of signal.

### ```2. Living Room```
> **Location**: Living room only

> **Justification**: Temperature transition similar towards outdoor condition

### ```3. Bathroom```
> **Location**: Bathroom only

> **Justification**: Humidity forms an isolated cluster with low overall correlation (signified by pale red color) to all other indoor rooms.

### ```4. Outdoor```
> **Location**: Outside only

> **Justification**: Low correlation with every other location, either in terms of temperature or humidity.

### Apply Grouping Using Average

In [ ]:
# Define the grouping using number codes taken from UCI Appliances Energy page
# For reference: https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction
groups = {
    "indoor": [1, 3, 4, 7, 8, 9],
    "liv_room": [2],
    "bathroom": [5],
    "outdoor": [6],
}

for name, indices in groups.items():
    # Mean Temperature
    temp_cols = [f"T{i}" for i in indices]
    energy[f"T_{name}"] = energy[temp_cols].mean(axis=1)

    # Mean Humidity
    rh_cols = [f"RH_{i}" for i in indices]
    energy[f"RH_{name}"] = energy[rh_cols].mean(axis=1)

In [ ]:
# Drop old, unaggregated features
for i in range (9):
    energy = energy.drop([f'T{i+1}',  f'RH_{i+1}'], axis = 1)

In [ ]:
# Visualize changes
energy.head(3)